# Evaluación y Validación de Modelos para Predicción de Diabetes

Comparación de Regresión Logística y Random Forest utilizando el Dataset de PLD.

## 1. Clases y funciones



In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pandas import DataFrame
from scipy import stats
from scipy.stats import ttest_rel
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style('whitegrid')


In [ ]:

class ModelEvaluator:
    def __init__(self, name, model):
        self.name = name
        self.model = model

    def fit(self, X_train, y_train):
        self.model.fit(X_train, y_train)

    def predict(self, X, threshold=0.5):
        p = self.model.predict_proba(X)[:, 1]
        y = (p >= threshold).astype(int)
        return y, p

    def metrics(self, y_true, y_pred, y_prob):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return {
            'Model': self.name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred),
            'Recall': recall_score(y_true, y_pred),
            'F1': f1_score(y_true, y_pred),
            'ROC_AUC': roc_auc_score(y_true, y_prob),
            'Specificity': tn / (tn + fp)
        }

    def find_best_threshold(self, y_true, proba):
        best = (0, 0.90, 0.89)  #Umbrales mínimos:(t=0.5, f1 0.9, Precisión 0.89)
        for t in np.arange(.1, .95, .01):
            f1 = f1_score(y_true, (proba >= t).astype(int))
            precision = precision_score(y_true, (proba >= t).astype(int))
            if f1 > best[1] and precision > best[2]:
                best = (t, f1, precision)
        if best[0] == 0:
            print(f"No se encotnró mejor valor para el modelo {self.name}, se deja threshold con valor 0.5")
            best = (0.5, 0, 0)
        return best


In [ ]:

from sklearn.model_selection import cross_val_score


class PLDEvaluatorModels:
    def __init__(self, datasetPath: str = '../data/resultados_pld.csv',
                 target: str = 'diabetes_target',
                 models_list=None, test_size: float = 0.2):
        if models_list is None:
            models_list = ['LogisticRegression', 'RandomForest']
        self.datasetPath = datasetPath
        self.df = pd.read_csv(self.datasetPath)
        self.target = target
        self.models_list = models_list
        self.test_size = test_size
        self.X = None
        self.y = None

        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.models = {}

        self.results_df = DataFrame()
        self.artifacts = {}

        self.result_test_ab = pd.DataFrame()

    def preprocess(self):
        if self.datasetPath is None:
            print(f"Dataset not loaded")
            return

        self.X = self.df.drop(columns=[self.target])
        self.y = self.df[self.target]

        cat = self.X.select_dtypes(include=['object', 'str']).columns.tolist()
        nan = self.X.select_dtypes(exclude=['object', 'str']).columns.tolist()
        num = [c for c in self.X.columns if c not in cat]

        prep = ColumnTransformer([
            ('num', StandardScaler(), num),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
        ])
        if self.models_list is not None:
            self.models = {
                'LogisticRegression': Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=3000))]),
                'RandomForest': Pipeline(
                    [('prep', prep), ('clf', RandomForestClassifier(n_estimators=300, random_state=42))]),
            }

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.X, self.y,
                                                                                test_size=self.test_size,
                                                                                stratify=self.y, random_state=42)

    def run_models(self):

        results = []
        for name, pipe in self.models.items():
            ev = ModelEvaluator(name, pipe)
            ev.fit(self.X_train, self.y_train)
            _, prob = ev.predict(self.X_test)

            threshold, best_recall, best_prec = ev.find_best_threshold(self.y_test, prob)
            print(f"Model: {name} Treshold: {threshold}, Best f1 score: {best_recall}, Best Precision: {best_prec}")
            pred, prob = ev.predict(self.X_test, threshold=threshold)

            m = ev.metrics(self.y_test, pred, prob)
            scoring = {
                'accuracy': 'accuracy',  # Las métricas como accuracy no generan division zero
                'prec': make_scorer(precision_score, average='macro', zero_division=0),
                'f1': make_scorer(f1_score, average='macro', zero_division=0),
                'roc_auc': 'roc_auc',
                'recall': make_scorer(recall_score, average='macro', zero_division=0),

            }
            cv = cross_validate(pipe, self.X, self.y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                scoring=scoring
                                )

            scores = cross_val_score(pipe, self.X, self.y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                     scoring='recall')
            mean_score = np.round(np.mean(scores), 2)
            std_f1 = np.std(scores, ddof=1)
            n = len(scores)
            t_value = stats.t.ppf(1 - 0.025, df=n - 1)
            margin_error = t_value * (std_f1 / np.sqrt(n))
            ci_lower = mean_score - margin_error
            ci_upper = mean_score + margin_error

            m['CV_ROC_AUC_MEAN'] = cv['test_roc_auc'].mean()
            m['CV_ROC_AUC_STD'] = cv['test_roc_auc'].std()
            m['threshold'] = threshold
            m['intervalo_confianza'] = f"({ci_lower:4f}, {ci_upper:4f})"
            m["margin_error"] = margin_error
            m['standard_deviation'] = std_f1

            results.append(m)
            self.artifacts[name] = {'pred': pred, 'prob': prob, 'cv': cv}
            with open(f'../models/pld_model_{name}.pkl', 'wb') as f:
                pickle.dump(ev.model, f)

        self.results_df = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False)
        return self.results_df

    def plot_confusionmatrix(self):

        for model in ['LogisticRegression', 'RandomForest']:
            cm = confusion_matrix(self.y_test, self.artifacts[model]['pred'])
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title(f'Matriz de Confusion - {model}')
            plt.show()

    def plot_roc_auc(self):

        plt.figure(figsize=(8, 6))
        for model in ['LogisticRegression', 'RandomForest']:
            p = self.artifacts[model]['prob']
            fpr, tpr, _ = roc_curve(self.y_test, p)
            auc = roc_auc_score(self.y_test, p)
            plt.plot(fpr, tpr, label=f'{model} AUC={auc:.3f}')

        plt.plot([0, 1], [0, 1], 'k--')
        plt.legend()
        plt.title('ROC-AUC Comparativo')
        plt.show()

    def test_AB(self):
        rf = self.artifacts['RandomForest']['cv']['test_roc_auc']

        lr = self.artifacts['LogisticRegression']['cv']['test_roc_auc']
        stat2, p_value2 = ttest_rel(rf, lr)
        data_test2 = pd.DataFrame(
            {"Prueba": ['A/B Test RandomForest vs LogisticRegression'], 't': [stat2], 'p-value': [p_value2]})
        self.result_test_ab = pd.concat([self.result_test_ab, data_test2], ignore_index=True)

        return self.result_test_ab

    def plot_tablero(self):
        table_df = self.results_df.copy()
        numeric_cols = table_df.select_dtypes(include=np.number).columns
        table_df[numeric_cols] = table_df[numeric_cols].round(4)

        fig, ax = plt.subplots(1, 2, figsize=(14, 10))
        self.results_df.plot(x='Model', y=['Accuracy', 'Precision', 'Recall', 'F1'], kind='bar', ax=ax[0])
        self.results_df.plot(x='Model', y='ROC_AUC', kind='bar', ax=ax[1], color='green')
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(2, 1, figsize=(14, 10))
        self.results_df.plot(x='Model', y='CV_ROC_AUC_MEAN', kind='bar', ax=ax[0], color='orange')
        ax[1].axis('off')
        ax[1].table(
            cellText=table_df.values,
            colLabels=table_df.columns,
            loc='center'
        )
        plt.tight_layout()
        plt.show()



## 2. Dataset - Diabetes Prediction Dataset.

El Dataset utilizado para realizar la comparativa se tomó del Dataset público de Kaggle con Licencia Data files © Original Authors. El dataset se puede descargar de la [liga](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset)

Para realizar la descarga desde un entorno Python se puede usar el código:

```
import kagglehub
# Download latest version
path = kagglehub.dataset_download("iammustafatz/diabetes-prediction-dataset")
print("Path to dataset files:", path)
```

La estructura del Dataset es la siguiente:

In [ ]:
path = '../data/archive/resultados.csv'
evaluatorModel = PLDEvaluatorModels(datasetPath=path, target='IS_FRAUD')
evaluatorModel.df.head()

## 3. Separación Datos de prueba y selección de modelos a probar.

Para comenzar con el entrenamiento, primero es necesario seleccionar los datos que se usarán para el entrenamiento, así como los campos que se usarán como objetivo y cuales para identificar patrones dentro de los modelos.

Una vez identificados los campos a utilizar se realizan los ajustes para estandarizar los datos y normalizarlos de tal manera que todos queden dentro de una escala adecuada, así como de solo usar datos que se puedan procesar con el modelo.

Posteriormente, se seleccionarán los modelos de ML que se utilizarán para realizar el comparativo:

- LogisticRegression
- RandomForest


In [ ]:

evaluatorModel.preprocess()

## 4. Experimentación

Se inicializa el entrenamiento y la experimentación con cada uno de los modelos; se recuperan métricas para el cálculo de curva ROC y el cálculo de AUC.


In [ ]:

results_df = evaluatorModel.run_models()
display(results_df)

## 5. Matriz de Confusión

Se realiza un comparativo entre los modelos a través de us Matriz de confusión para validar los falsos positivos y falsos negativos que arroja cada uno de los modelos.

In [ ]:
evaluatorModel.plot_confusionmatrix()

## 6. ROC-AUC

Del mismo modo se muestra la gráfica ROC-AUC para comparar cada uno de los modelos.

Como se puede observar los modelos de Logistic Regression y Random Forest obtuvieron un mayor puntaje.

In [ ]:
evaluatorModel.plot_roc_auc()

## 7. Prueba A/B

Para la comparación entre modelos se hará uso de pruebas A/B entre los modelos:

- RandomForest
- LogisticRegression


In [ ]:
prueba_ab = evaluatorModel.test_AB()
display(prueba_ab)

## 8. Tablero

Se presentan los resultados de la comparación de los 3 modelos a través de un tablero en donde se muestran:

- Métricas comparativas:
    -  Accuracy
    - Precision
    - Recall
    - F1-score
- ROC_AUC
- Media ROC_AUC
- Tabla comparativa de métricas


In [ ]:
evaluatorModel.plot_tablero()

## 9. Selección del mejor modelo.

Para determinar el mejor modelo se hace una comparativa entre la métrica F1-Score de cada modelo ya que esta es la métrica que mejor balance tiene entre los resultados. Además se calcula el umbral de cada modelo  para definir cual es el mejor modelo.

In [ ]:

best_model = evaluatorModel.results_df.iloc[0]
display(best_model)
